In [ ]:
!pip install transformers datasets sentencepiece evaluate rouge_score
!pip install rouge-score

import pandas as pd
import numpy as np
import torch
import random
import evaluate

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq
)

from transformers import Seq2SeqTrainingArguments

from transformers import Seq2SeqTrainer

from tqdm import tqdm

import matplotlib.pyplot as plt

from transformers import set_seed

from nltk.translate.bleu_score import corpus_bleu

from rouge_score import rouge_scorer

seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

#Read the training set first
train = pd.read_json("/kaggle/input/datasets/zelenezhong/clickbait-spoiler-dataset/train.jsonl", lines=True)

In [ ]:
#See the first few rows for the training set

train.head()

In [ ]:
#Also, see the column names of the train dataset
train.columns

Based on the dataset description, 14 variables are available. The target variable is spoiler

In [ ]:
#Read the test dataset as well as the validation set
test = pd.read_json("/kaggle/input/datasets/zelenezhong/clickbait-spoiler-dataset/test.jsonl", lines=True)

val = pd.read_json("/kaggle/input/datasets/zelenezhong/clickbait-spoiler-dataset/val.jsonl", lines=True)

#See what variables are available in the test dataset
test.columns

For the input variables, we mainly focus on postText, targetTitle, and targetParagraphs. Then we need to change the targetParagraphs from list type to str type.

In [ ]:
#Here, we consider to limit the number of paragraphs we use. In this case, if the paragraphs length is smaller than 6, we can use all of them
#If it is longer than 6, then we choose the first four and the last two paragraphs.
#See the original version without improvement
def combine_paragraphs(paragraphs):
    if len(paragraphs) <= 6:
        return " ".join(paragraphs)
    return " ".join(paragraphs[:4] + paragraphs[-2:])


train["context"] = train["targetParagraphs"].apply(combine_paragraphs)

val["context"] = val["targetParagraphs"].apply(combine_paragraphs)

test["context"] = test["targetParagraphs"].apply(combine_paragraphs)

Similarly, we need to do the same for postText

In [ ]:
def combine_text(x):
    if isinstance(x, list):
        return " ".join(x)
    return x

train["postText"] = train["postText"].apply(combine_text)

val["postText"] = val["postText"].apply(combine_text)

test["postText"] = test["postText"].apply(combine_text)

Then we need to transform the format of the spoiler

In [ ]:
def process_spoiler(spoiler):
    return " ".join(spoiler)

train["target_text"] = train["spoiler"].apply(process_spoiler)

val["target_text"] = val["spoiler"].apply(process_spoiler)

So the type of spoiler changes. Next, we create the BART input

In [ ]:
def create_input(row):

    return (
        "generate spoiler: "
        + row["postText"]
        + " title: "
        + row["targetTitle"]
        + " context: "
        + row["context"]
    )


train["input_text"] = train.apply(create_input, axis=1)

val["input_text"] = val.apply(create_input, axis=1)

test["input_text"] = test.apply(create_input, axis=1)

Next, before the formal tokenizer is applied, we need to check and confirm how many pieces of text exceed the maximum length allowed by BART.

In [ ]:
#Load the tokenizater first
tokenizer = AutoTokenizer.from_pretrained(
    "facebook/bart-base"
)

#check the length
train_lengths = train["input_text"].apply(
    lambda x: len(tokenizer(x)["input_ids"])
)

print(train_lengths.describe())

In [ ]:
(train_lengths > 1024).sum()

In [ ]:
#We need to check the percentage that exceed max length
(train_lengths > 1024).mean()

This means that 0.375% of the training samples exceed the maximum input length of 1024 tokens that BART can handle.

Then, we can create a HuggingFace Dataset.

In [ ]:
max_input_length = 1024

train_dataset = Dataset.from_pandas(
    train[["input_text", "target_text"]]
)

val_dataset = Dataset.from_pandas(
    val[["input_text", "target_text"]]
)

test_dataset = Dataset.from_pandas(
    test[["input_text"]]
)

#The generated spoiler is limited to a maximum of 64 tokens.
max_target_length = 64

target_lengths = train["target_text"].apply(
    lambda x: len(tokenizer(x)["input_ids"])
)

target_lengths.describe()

In [ ]:
(target_lengths > 64).mean()

Only about 5% of the spoilers exceed the token length of 64, so set it to 64 is reasonable.

Then, we need to load the BART model.

In [ ]:
model_name = "facebook/bart-base"


model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name
)

Then, we need to write the tokenizer function

In [ ]:
#The max_input_length and max_target_length are both settled before

def tokenize(batch):

    inputs = tokenizer(
        batch["input_text"],
        max_length=max_input_length,
        truncation=True
    )


    targets = tokenizer(
        batch["target_text"],
        max_length=max_target_length,
        truncation=True
    )


    inputs["labels"] = targets["input_ids"]

    return inputs

In [ ]:
train_dataset = train_dataset.map(
    tokenize,
    batched=True
)


val_dataset = val_dataset.map(
    tokenize,
    batched=True
)

Next, we need the Data Collator. During the training process, it combines multiple pieces of data into one batch and performs dynamic processing.

In [ ]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

Next, set up TrainingArguments

In [ ]:
training_args = Seq2SeqTrainingArguments(

    output_dir="./bart_base_results",

    eval_strategy="epoch",
    save_strategy="epoch",

    learning_rate=3e-5,

    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,

    num_train_epochs=5,

    weight_decay=0.01,

    predict_with_generate=True,

    load_best_model_at_end=True,

    fp16=True,

    logging_steps=50
)

Then, create the trainer

In [ ]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator
)

Then, apply the training dataset to the model

In [ ]:
trainer.train()

Then, see the output for this

In [ ]:
#See the evaluate score
trainer.evaluate()

Then, we would like to evaluate how this performs in the validation dataset. We use BLEU, ROUGE-L, and METEOR.

BLEU

In [ ]:
predictions = []

model.eval()

for i in tqdm(range(len(val_dataset))):

    inputs = {
        k: torch.tensor(v).unsqueeze(0).to(model.device)
        for k, v in val_dataset[i].items()
        if k in ["input_ids", "attention_mask"]
    }
    with torch.no_grad():

        generated_ids = model.generate(
            **inputs,
            max_length=64,
            num_beams=4
        )

    pred = tokenizer.decode(
        generated_ids[0],
        skip_special_tokens=True
    )

    predictions.append(pred)

In [ ]:
references = [
    x if isinstance(x, list) else [x]
    for x in val["spoiler"]
]

bleu = evaluate.load("bleu")
bleu_score = bleu.compute(
    predictions=predictions,
    references=references
)
print("BLEU:", bleu_score["bleu"])

ROUGE-L

In [ ]:
scorer = rouge_scorer.RougeScorer(
    ["rougeL"],
    use_stemmer=True
)

scores = []

for pred, ref in zip(predictions, references):
    score = scorer.score(ref[0], pred)
    scores.append(score["rougeL"].fmeasure)

rougeL = sum(scores) / len(scores)

print("ROUGE-L:", rougeL)

METEOR

In [ ]:
meteor = evaluate.load("meteor")

score = meteor.compute(
    predictions=predictions,
    references=references
)

print(score["meteor"])

Take a manual inspection of the spoiler effect generated by BART.

In [ ]:
for i in range(5):

    sample = val.iloc[i]

    inputs = tokenizer(
        sample["input_text"],
        max_length=1024,
        truncation=True,
        return_tensors="pt"
    ).to(model.device)


    generated_ids = model.generate(
        **inputs,
        max_length=64,
        num_beams=4
    )


    pred = tokenizer.decode(
        generated_ids[0],
        skip_special_tokens=True
    )


    print("="*50)
    print("TRUE:")
    print(sample["target_text"])

    print("PRED:")
    print(pred)

Then, we use the model on the test dataset

In [ ]:
test_inputs = tokenizer(
    test["input_text"].tolist(),
    max_length=1024,
    truncation=True,
    padding=True,
    return_tensors="pt"
)

test_inputs = {
    k: v.to(model.device)
    for k, v in test_inputs.items()
}

In [ ]:
#We can also try to change the hyperparameter to improve the prediction.
model.eval()

test_predictions = []

batch_size = 4

for i in tqdm(range(0, len(test), batch_size)):

    batch_texts = test["input_text"].iloc[i:i+batch_size].tolist()

    inputs = tokenizer(
        batch_texts,
        max_length=1024,
        truncation=True,
        padding=True,
        return_tensors="pt"
    ).to(model.device)


    with torch.no_grad():

        generated_ids = model.generate(
            **inputs,
            max_length=64,
            num_beams=4
        )


    decoded = [
        tokenizer.decode(
            ids,
            skip_special_tokens=True
        )
        for ids in generated_ids
    ]

    test_predictions.extend(decoded)


In [ ]:
len(test_predictions)

In [ ]:
for i in range(5):
    print("Prediction", i, ":")
    print(test_predictions[i])
    print()

In [ ]:
#Then we have the submission

submission_task2 = pd.DataFrame({
    "id": test["id"],
    "spoiler": test_predictions
})

#Check the head of the submission
submission_task2.head()

In [ ]:
submission_task2.to_csv(
    "prediction_task2.csv",
    index=False
)